<a href="https://colab.research.google.com/github/kdkim2000/RAG2026/blob/main/%5B%EC%8B%A4%EC%8A%B5%5D_4_%EB%B2%A1%ED%84%B0_%EB%8D%B0%EC%9D%B4%ED%84%B0%EB%B2%A0%EC%9D%B4%EC%8A%A4_%EA%B8%B0%EB%B0%98_RAG_%EC%96%B4%ED%94%8C%EB%A6%AC%EC%BC%80%EC%9D%B4%EC%85%98_%EB%A7%8C%EB%93%A4%EA%B8%B0_ipynb%EC%9D%98_%EC%82%AC%EB%B3%B8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# [실습] 벡터 데이터베이스 기반 RAG 어플리케이션 만들기

RAG는 Retrieval-Augmented Generation (RAG) 의 약자로,   
LLM의 작동 과정에 검색을 결합하여 답변 성능을 높이는 어플리케이션입니다.   


이번 실습에서는 뉴스 검색 데이터를 이용한 RAG를 수행해 보겠습니다.    
코랩 GPU 사용을 위한 설정이 필요합니다.

### <필수> 실습을 진행하기 전, GPU를 T4로 설정해 주세요!

## 라이브러리 설치  

랭체인 관련 라이브러리와 벡터 데이터베이스 라이브러리를 설치합니다.  
<br>

`sentence_transformers`: 트랜스포머 계열의 공개 임베딩 모델을 사용할 수 있습니다.    
`langchain_chroma`: ChromaDB를 이용해 벡터 데이터베이스를 구성합니다.

In [ ]:
%pip install dotenv langchain_huggingface transformers sentence_transformers jsonlines langchain langchain-openai langchain-community langchain_chroma -q

코랩에서 실행 시, Restart 메시지가 나타납니다. 런타임을 재시작하여 패키지를 정리합니다.

## LLM과 임베딩 모델 구성하기   

이번 실습에서는 LLM 모델과 함께 임베딩 모델이 필요합니다.   
임베딩 모델은 텍스트를 벡터로 변환하며,    
이후 결과를 벡터 DB에 저장해 검색할 수 있습니다.

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv(override=True)

if os.environ.get('OPENAI_API_KEY'):
    print('OpenAI API 키 확인')

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-5.1", reasoning_effort='low')

OpenAI의 `text-embedding-3-large` 는 빠른 속도로 연산이 가능하나, 비용이 발생하며 온라인 모델입니다.   
이에 따라, 폐쇄망/온프레미스 환경에서는 공개 임베딩 모델을 사용하여 구현해야 합니다.

In [ ]:
from langchain_openai import OpenAIEmbeddings
openai_embeddings = OpenAIEmbeddings(model='text-embedding-3-large')

허깅페이스에 게시된 공개 모델을 불러옵니다.   
오픈 임베딩 모델에서 중요한 파라미터는 다음과 같습니다.

- 파라미터 수 : 큰 임베딩 모델의 크기는 LLM에 육박합니다. GPU를 고려하여 선택합니다.
- Max Tokens: 임베딩 모델의 최대 토큰보다 큰 데이터를 입력하면, 앞부분만을 이용해 계산하게 되므로 적절한 검색이 되지 않을 수 있습니다.
- 임베딩 차원: 큰 차원의 벡터를 생성하는 임베딩 모델은 검색 속도가 감소합니다.

현재 한국어 데이터를 임베딩하기 위해 자주 사용하는 모델은 아래와 같습니다.


- Qwen/Qwen-3-Embedding (0.6B, 4B, 8B, 32768 토큰 제한)    
알리바바 클라우드의 Qwen 모델을 개량하여 만든 모델입니다.    
가장 최신 모델로, BGE-M3와의 성능 비교가 치열합니다.


Qwen 3 임베딩 모델을 불러옵니다.

In [ ]:
# 터미널에서 아래 코드를 실행해도 됨
# hf download Qwen/Qwen3-Embedding-0.6B --local-dir ./embedding

In [ ]:
from sentence_transformers import SentenceTransformer
import torch

# HuggingFace 임베딩 주소 지정하기
# intfloat/multilingual-e5-small , baai/bge-m3, 등의 주소를 입력하여 지정
# GPU에 여유가 있다면 Qwen3 Embedding의 큰 사이즈 (4B, 8B)

model_name = 'Qwen/Qwen3-Embedding-0.6B'
#실제 주소: https://huggingface.co/Qwen/Qwen3-Embedding-0.6B

# CPU 설정으로 모델 불러오기
emb_model = SentenceTransformer(model_name, device='cpu',model_kwargs={'torch_dtype':torch.bfloat16})

# 로컬 폴더에 모델 저장하기
emb_model.save('./embedding')

# 모델 메모리에서 삭제
del emb_model
import gc
gc.collect()

print("임베딩 모델 저장 완료!")

파일 시스템에 저장한 오픈 모델은 HuggingFaceEmbeddings로 불러옵니다.

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

# 허깅페이스 포맷의 임베딩 모델 불러오기
open_embeddings = HuggingFaceEmbeddings(model_name= './embedding',
                                  model_kwargs={'device':'cuda',
                                                'model_kwargs':{'torch_dtype':torch.bfloat16}}) # gpu 사용하기

# GPU 로드된 것 확인
print("임베딩 모델 GPU 로드 완료")

RAG를 하기 전, 비교를 위해 LLM에게 질문해 보겠습니다.

In [ ]:
# Test
llm.invoke("도메인 특화 언어 모델이란 무엇입니까? 어떤 예시가 있나요?")

## 데이터 준비하기    
네이버 API를 통해, 검색어에 대한 뉴스 기사 링크를 가져오겠습니다.    


In [ ]:
import requests
def get_naver_news_links(query, num_links=100):
    """
    query와 num_links를 입력받아 네이버 검색 수행, 네이버 뉴스 URL의 기사만 수집
    """

    url = f"https://openapi.naver.com/v1/search/news.json?query={query}&display={num_links}&sort=sim"
    # 최대 100개의 결과를 표시
    headers = {
        'X-Naver-Client-Id': 'Ko6yIqbV2TOHq9rPH8tu',
        'X-Naver-Client-Secret': 'BvqX8mNtHu'
    }

    response = requests.get(url, headers=headers)
    result = response.json()
    # 특정 링크 형식만 필터링
    filtered_links = []
    for item in result['items']:
        link = item['link']
        if "n.news.naver.com/mnews/article/" in link:
            # 네이버 뉴스 스타일만 모으기
            filtered_links.append(link)

    # 결과 출력
    print(query, ':', len(filtered_links), 'Example:', filtered_links[0])
    # for link in filtered_links:
    #     print(link)

    return filtered_links

filtered_links = []
for topic in ['도메인 특화 언어모델', 'OpenAI', 'GPT', '구글', '가전제품', '넷플릭스']:
    filtered_links += get_naver_news_links(topic, 100)
print('Total Articles:', len(filtered_links))
print('Total Articles(Without Duplicate):',len(list(set(filtered_links))))
filtered_links = list(set(filtered_links))

## LangChain Document Loaders

LangChain의 `document_loaders`는 다양한 형식의 파일을 불러올 수 있습니다.   
[https://python.langchain.com/docs/integrations/document_loaders/ ]    

Web URL로부터 페이지를 로드하는 기본 파서인 `WebBaseLoader`를 사용합니다.   

In [ ]:
# # Jupyter 분산 처리를 위한 설정 (코랩에서는 불필요)
# import nest_asyncio

# nest_asyncio.apply()

In [ ]:
import bs4
from langchain_community.document_loaders import WebBaseLoader

async def get_news_documents(links):
    loader = WebBaseLoader(
        web_paths=links,
        bs_kwargs={'parse_only':bs4.SoupStrainer(class_=("newsct", "newsct-body"))},
                                # newsct, newsct-body만 추출 : 네이버 뉴스 포맷 HTML 요소

        requests_per_second = 10, # 1초에 10개 요청 보내기
        show_progress = True # 진행 상황 출력
    )
    # docs = loader.load() # 기본 코드

    docs = []

    async for doc in loader.alazy_load(): # 순차적 로드 대신 비동기 처리
        docs.append(doc)
    return docs

docs = await get_news_documents(filtered_links)

In [ ]:
docs[12]

크롤링 결과에는 불필요한 문자가 많이 포함되어 있습니다.    
전처리를 통해 이를 제거합니다.

In [ ]:
import re

def preprocess(docs):
    noise_texts = [
        '''구독중 구독자 0 응원수 0 더보기''',
        '''쏠쏠정보 0 흥미진진 0 공감백배 0 분석탁월 0 후속강추 0''',
        '''댓글 본문 요약봇 본문 요약봇''',
        '''도움말 자동 추출 기술로 요약된 내용입니다. 요약 기술의 특성상 본문의 주요 내용이 제외될 수 있어, 전체 맥락을 이해하기 위해서는 기사 본문 전체보기를 권장합니다. 닫기''',
        '''텍스트 음성 변환 서비스 사용하기 성별 남성 여성 말하기 속도 느림 보통 빠름''',
        '''이동 통신망을 이용하여 음성을 재생하면 별도의 데이터 통화료가 부과될 수 있습니다. 본문듣기 시작''',
        '''닫기 글자 크기 변경하기 가1단계 작게 가2단계 보통 가3단계 크게 가4단계 아주크게 가5단계 최대크게 SNS 보내기 인쇄하기''',
        'PICK 안내 언론사가 주요기사로선정한 기사입니다. 언론사별 바로가기 닫기',
        '응원 닫기',
        '구독 구독중 구독자 0 응원수 0 ',
    ]

    def clean_text(doc):
        text = doc.page_content
        # 탭과 개행문자를 공백으로 변환
        text = text.replace('\t', ' ').replace('\n', ' ')

        # 연속된 공백을 하나로 치환
        text = re.sub(r'\s+', ' ', text).strip()

        # 여러 구분자를 한번에 처리
        split_markers = [
            '구독 해지되었습니다.',
            '구독 메인에서 바로 보는 언론사 편집 뉴스 지금 바로 구독해보세요!'
        ]
        for marker in split_markers:
            parts = text.split(marker)
            if len(parts) > 1:
                if marker == '구독 해지되었습니다.':
                    text = parts[1]  # 뒷부분 사용
                else:
                    text = parts[0]  # 앞부분 사용

        # 노이즈 텍스트 제거
        for noise in noise_texts:
            text = text.replace(noise, '')

        # 연속된 공백을 하나로 치환
        text = re.sub(r'\s+', ' ', text).strip()
        doc.page_content = text
        return doc

    preprocessed_docs = []
    for doc in docs:

        # 텍스트 정제
        doc= clean_text(doc)
        preprocessed_docs.append(doc)

    return preprocessed_docs

preprocessed_docs = preprocess(docs)


In [ ]:
preprocessed_docs[2]

불러온 텍스트 데이터는 파일로 저장할 수 있습니다.

In [ ]:
# 불러온 document 저장하기

import jsonlines
def save_docs_to_jsonl(documents, file_path):
    with jsonlines.open(file_path, mode="w") as writer:
        for doc in documents:
            writer.write(doc.model_dump())

# jsonl 파일 불러오기
from langchain_core.documents import Document

def load_docs_from_jsonl(file_path):
    documents = []
    with jsonlines.open(file_path, mode="r") as reader:
        for doc in reader:
            documents.append(Document(**doc))
    return documents

In [ ]:
# 저장
save_docs_to_jsonl(preprocessed_docs, "docs.jsonl")

## Chunking: 청크 단위로 나누기   



전처리가 완료된 docs를 chunk 단위로 분리합니다.
`chunk_size`와 `chunk_overlap`을 이용해 청크의 구성 방식을 조절할 수 있습니다.

Chunk Size * K(검색할 청크의 수) 의 결과가 Context의 길이가 됩니다.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
# 0~1000, 800~1800, 1600~2600, ...
chunks = text_splitter.split_documents(preprocessed_docs)
print(len(chunks))

## Vector DB 구성하기

구성된 청크를 ChromaDB 벡터 데이터베이스에 로드합니다.   

In [ ]:
from langchain_chroma import Chroma

Chroma().delete_collection() # (메모리에 저장하는 경우) 기존 데이터 삭제

# DB 구성하기
db = Chroma(embedding_function=openai_embeddings,
            persist_directory="./chroma_OpenAI",
            # 파일 시스템에 저장 (생략시 메모리에 저장)

            collection_name='Web', # 식별 이름

            collection_metadata={'hnsw:space':'l2'},
            # l2 메트릭 설정(기본값, cosine, mmr 로 변경 가능)
            )

DB에 document를 추가합니다.    
OpenAI 임베딩은 30만 토큰 동시 처리 제한이 있어, 나눠서 전달합니다.

In [ ]:
from tqdm import tqdm
import time
print(len(chunks))
# 300,000 토큰 제한

# 20개씩 추가
for i in tqdm(range(0, len(chunks), 20)):
    db.add_documents(chunks[i:min(i+20, len(chunks))])
    time.sleep(5)

db로부터 retriever를 구성합니다.

In [ ]:
# Top 5 Search(기본값은 4)
retriever = db.as_retriever(search_kwargs={'k':5})

In [ ]:
context = retriever.invoke("도메인 특화 언어 모델")
context

위 검색 결과를 전처리하여, LLM의 프롬프트로 넣기 위한 함수를 구성합니다.

In [ ]:
from typing import Iterable, Sequence
from xml.sax.saxutils import escape
from langchain_core.documents import Document


def format_docs(
    docs: Iterable[Document],
    metadata_keys: Sequence[str] = ("source",),
) -> str:
    """
    List[Document] -> XML 직렬화 문자열.

    - 청크 경계: <document index="N"> 태그로 명시
    - 메타데이터: metadata_keys 에 지정한 키만 <meta>로 포함 (누락 키는 자동 생략)
    - 본문: XML 특수문자 escape (본문에 '<', '>' 가 있어도 경계 유지)
    """
    parts: list[str] = ["<documents>"]
    for i, doc in enumerate(docs, start=1):
        parts.append(f'  <document index="{i}">')
        for key in metadata_keys:
            value = doc.metadata.get(key)
            if value is None:
                continue
            parts.append(
                f'    <meta name="{escape(str(key))}">{escape(str(value))}</meta>'
            )
        parts.append(f"    <content>{escape(doc.page_content)}</content>")
        parts.append("  </document>")
    parts.append("</documents>")
    return "\n".join(parts)


print(format_docs(context, metadata_keys=("source", "page", "section")))

구성한 format_docs 함수는 이후 체인에 포함합니다.

## Prompting

RAG를 위한 간단한 프롬프트를 작성합니다.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
prompt = ChatPromptTemplate([
    ("user", '''당신은 QA(Question-Answering)을 수행하는 Assistant입니다.
다음의 Context를 이용하여 Question에 답변하세요.
정확한 답변을 제공하세요.
만약 모든 Context를 다 확인해도 정보가 없다면,
"정보가 부족하여 답변할 수 없습니다."를 출력하세요.
---
Context: {context}
---
Question: {question}''')])

prompt.pretty_print()

## Chain

RAG를 수행하기 위한 Chain을 만듭니다.

RAG Chain은 프롬프트에 context와 question을 전달해야 합니다.    
Question을 입력받아, Context를 함께 프롬프트에 전달합니다.

In [ ]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    # retriever : question을 받아서 context 검색: document 반환
    # format_docs : document 형태를 받아서 텍스트로 변환
    # RunnablePassthrough(): 체인의 입력을 그대로 저장
    | prompt
    | llm
    | StrOutputParser()
)

In [ ]:
rag_chain.invoke("도메인 특화 언어 모델이란 무엇입니까? 어떤 예시가 있나요?")

In [ ]:
rag_chain.invoke("인공지능의 최근 발전 방식은? 관련 링크도 보여주세요")

In [ ]:
rag_chain.invoke("알리바바의 언어 모델 이름은?")

assign()을 이용하면, 체인의 결과를 받아 새로운 체인에 전달하고, 그 결과를 가져옵니다.

In [ ]:
# assign : 결과를 받아서 새로운 인수 추가하고 원래 결과와 함께 전달

from langchain_core.runnables import RunnableParallel

rag_chain_from_docs = (
    prompt
    | llm
    | StrOutputParser()
)

rag_chain_with_source = RunnableParallel(
    context = retriever | format_docs, question = RunnablePassthrough()).assign(answer=rag_chain_from_docs)

rag_chain_with_source.invoke("인공지능의 최근 발전 방식은? 관련 링크도 보여주세요")

# retriever가 1번 실행됨
# retriever의 실행 결과를 rag_chain_from_docs 에 넘겨주기 때문에


이번에는 오픈 모델을 사용합니다.   
오픈 임베딩을 통해 구성한 DB와 원래 DB를 비교해 보겠습니다.

In [ ]:
open_db = Chroma(embedding_function=open_embeddings,
                           persist_directory="./chroma_open", # 별도 폴더에 저장
                           collection_name='Web', # 식별 이름
                           collection_metadata={'hnsw:space':'l2'}
                           )

# 20개씩 추가
for i in tqdm(range(0, len(chunks), 20)):
    open_db.add_documents(chunks[i:min(i+20, len(chunks))])


이후는 동일합니다.

In [ ]:
open_retriever = open_db.as_retriever(search_kwargs={'k':5})

In [ ]:
open_retriever.invoke("도메인 특화 언어 모델")

In [ ]:
rag_chain_open = (
    {"context": open_retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

예시 질문을 통해 정답과 검색 결과를 비교해 보겠습니다.

In [ ]:
questions = ["도메인 특화 언어 모델이란 무엇입니까? 어떤 예시가 있나요?",
             "인공지능의 최근 발전 방식은? 관련 링크도 보여주세요",
             "알리바바의 언어 모델 이름은?"]

In [ ]:
# Retriever 비교

contexts_hf = open_retriever.batch(questions)
contexts_openai = retriever.batch(questions)

In [ ]:
for i in range(len(questions)):
    print(f"-- Question:{questions[i]}")
    oai_chunks = '\n'.join([x.page_content[:50] for x in contexts_openai[i]])
    hf_chunks = '\n'.join([x.page_content[:50] for x in contexts_hf[i]])
    print(f"OpenAI: \n{oai_chunks}")
    print(f"Hf: \n{hf_chunks}")
    print('----------')



In [ ]:
# 최종 결과 비교
oai_results = rag_chain.batch(questions)
hf_results = rag_chain_open.batch(questions)

In [ ]:
for i in range(len(questions)):
    print(f"-- Question:{questions[i]}")
    print(f"OpenAI: \n{oai_results[i]}")
    print('--')
    print(f"Hf: \n{hf_results[i]}")
    print('----------')


## Resources

### Embedding 모델 요약

- **(~2024) BERT 기반의 모델:** 문맥상의 '평균'을 활용하여 임베딩을 수행합니다. 대표적인 예시로는 BGE-M3와 OpenAI Embedding이 있습니다.
- **(2025~) LLM 기반의 모델:** LLM이 이해한 문맥값을 기반으로 임베딩을 생성합니다. Qwen 3 Embedding이 이 범주에 속합니다.